# ThinkingBox in miniature: simulating the reliability gap

Microsoft + Hugging Face's ThinkingBox (arXiv 2608.19741, "One Success Isn't Reliability") grades agents
on the **final database state**, not the chat transcript: 507 business-workflow tasks, each run **20 times**
on a freshly reset backend, reported as pass@1 / pass@20 / observed-20-of-20.

This notebook does three things:
1. Simulates repeated runs to reproduce the capability-vs-dependability collapse
   (Claude Opus 5: 66.50% pass@1 -> 47.53% observed-20-of-20; Kimi-K3: 57.37% -> 17.60%).
2. Implements a toy deterministic state-diff judge: fixture DB in, agent's tool-call trace applied,
   required end state + forbidden side effects checked. Buckets failures the ThinkingBox way
   (wrong values / unintended side effects / missed changes).
3. Computes expected cost of a *dependable* outcome when you retry until the check passes.


In [ ]:
import numpy as np

rng = np.random.default_rng(42)

def simulate(p_task, n_tasks=507, n_runs=20):
    """p_task: per-task latent single-run success probs. Returns dict of the three metrics."""
    draws = rng.random((n_tasks, n_runs)) < p_task[:, None]
    n_succ = draws.sum(axis=1)
    return {
        "pass@1": draws.mean(),                      # all attempts
        "pass@20": (n_succ >= 1).mean(),             # solved at least once
        "observed_20_of_20": (n_succ == n_runs).mean(),  # every attempt
    }

# Two-cluster ability mix, calibrated so simulated pass@1 lands near the reported values.
# NOTE: the sim reproduces the *collapse direction* (pass@20 >> observed-20-of-20),
# not exact magnitudes -- that needs per-task heterogeneity the summaries do not expose.
def two_cluster(p_easy, p_hard, frac_easy):
    n = 507
    p = np.where(rng.random(n) < frac_easy, p_easy, p_hard)
    return p

opus = simulate(two_cluster(0.88, 0.35, 0.60))
kimi = simulate(two_cluster(0.78, 0.28, 0.60))
print("Opus-like  :", {k: round(v, 4) for k, v in opus.items()})
print("Kimi-like  :", {k: round(v, 4) for k, v in kimi.items()})

# Sanity: simulated pass@1 must sit near the reported figures, and 20-of-20 must trail it hard.
assert 0.63 < opus["pass@1"] < 0.70   # reported: 66.50%
assert 0.54 < kimi["pass@1"] < 0.61   # reported: 57.37%
assert opus["observed_20_of_20"] < opus["pass@1"] - 0.10
assert kimi["pass@20"] > 0.90             # "solves 93.9% at least once" family of behavior
assert kimi["observed_20_of_20"] < 0.25  # yet dependable on a small fraction
print("gap OK: capability (pass@20) >> dependability (20-of-20), same as ThinkingBox")

## The state-diff judge

The ThinkingBox insight: don't grade the trajectory. Snapshot the DB before and after, and check:
(1) required field values, (2) no changes to unrelated records (side effects), (3) no missed changes.

Below: a toy `orders` + `customers` fixture, an agent that must move order #42 to `shipped`
and refund $50 to customer #7. We replay three tool-call traces: clean, corrupted (wrong value +
extra side effect, but *no tool error*), and a no-op. The corrupted one is exactly the silent-failure
class ThinkingBox says ends 67.24% of failed trials with no final tool error.

In [ ]:
def fixture():
    return {
        "orders": {42: {"status": "paid", "amount": 199.00}},
        "customers": {7: {"balance": 0.0}, 8: {"balance": 0.0}},
    }

def apply(db, calls):
    """calls: list of (table, key, field, value). Silent no-ops allowed, no exceptions."""
    for t, k, f, v in calls:
        if t in db and k in db[t]:
            db[t][k][f] = v

def judge(before, after, required, protected):
    """required: (table, key, field) -> value. protected: records that must not change."""
    missed, wrong = [], []
    for (t, k, f), v in required.items():
        if f not in after.get(t, {}).get(k, {}):
            missed.append((t, k, f))
        elif after[t][k][f] != v:
            wrong.append(((t, k, f), after[t][k][f], v))
    side_effects = []
    for t in after:
        for k in after[t]:
            for f in after[t][k]:
                is_required = (t, k, f) in required
                if not is_required and (t, k, f) in protected and after[t][k][f] != before[t][k][f]:
                    side_effects.append(((t, k, f), before[t][k][f], after[t][k][f]))
    return {"missed": missed, "wrong_values": wrong, "side_effects": side_effects}

REQUIRED = {("orders", 42, "status"): "shipped", ("customers", 7, "balance"): 50.0}
PROTECTED = {("orders", 42, "amount"), ("customers", 7, "balance"), ("customers", 8, "balance"),
               ("orders", 42, "status")}

traces = {
    "clean":     [("orders", 42, "status", "shipped"), ("customers", 7, "balance", 50.0)],
    "corrupted": [("orders", 42, "status", "shipped"), ("customers", 7, "balance", 500.0),
                  ("customers", 8, "balance", 100.0)],  # wrong refund + touches customer 8; no error
    "noop":      [],
}

for name, calls in traces.items():
    b4, af = fixture(), fixture()
    apply(af, calls)
    verdict = judge(b4, af, REQUIRED, PROTECTED)
    passed = not (verdict["missed"] or verdict["wrong_values"] or verdict["side_effects"])
    print(f"{name:>10}: {'PASS' if passed else 'FAIL'}  ->  {verdict}")

# The corrupted trace must fail on BOTH wrong value and side effect, silently.
b4, af = fixture(), fixture()
apply(af, traces["corrupted"])
v = judge(b4, af, REQUIRED, PROTECTED)
assert len(v["wrong_values"]) == 1 and len(v["side_effects"]) == 1 and not v["missed"]
print("assert: silent corruption caught as wrong-value + side-effect, no exception raised")

## Cost of a dependable outcome

If the benchmark's per-attempt pass probability is p, and you retry each task until the judge
passes (geometric retries), expected attempts = 1/p. A 'dependable' result in ThinkingBox terms
is roughly the observed-20-of-20 rate -- and the HF post's cost-per-dependable-task estimates
($6.80 GPT-5.4 / $7.45 GPT-6 Astra / $7.80 Opus 5.5) follow the same logic: the sticker price per
token is not the price of a finished, verified task.

In [ ]:
def dependable_cost(p_single, cost_per_attempt):
    """Expected cost to get one judge-verified success, retrying until pass."""
    assert 0 < p_single <= 1
    return cost_per_attempt / p_single

agents = {
    "Opus-like (p=0.665)": (0.665, 2.00),
    "Kimi-like (p=0.574)": (0.574, 0.80),
}
for name, (p, c) in agents.items():
    print(f"{name}: ${dependable_cost(p, c):.2f} expected per verified task")

assert dependable_cost(0.665, 2.00) < 2.00 / 0.60  # stronger agent => cheaper dependable outcome
assert abs(dependable_cost(1.0, 3.0) - 3.0) < 1e-9  # perfect agent pays sticker price
print("asserts pass")